# ⚙️ Notebook 3: Feature Engineering

Raw PaySim features are good but insufficient for high-performance fraud detection.
This notebook engineers domain-driven features that encode fraud signals from the data.

**Features we'll create:**
1. Balance discrepancy features (expected vs. actual)
2. Balance ratio features  
3. Zero-balance flags
4. Temporal features (hour, day, night/weekend)
5. Transaction type encodings
6. Amount-based features (log, round numbers, large transactions)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')
from src.data_loader import load_data
from src.feature_engineering import engineer_features, get_feature_names
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#0f1117',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#444', 'grid.color': '#2a2a2a',
})

DATA_PATH = '../data/PS_20174392719_1491204439457_log.csv'
df = load_data(DATA_PATH, sample_frac=0.2)
print(f"Loaded {len(df):,} rows")


In [ ]:
# ── Apply Feature Engineering Pipeline ────────────────────────────────────
original_cols = set(df.columns)
df_fe = engineer_features(df)
new_cols = [c for c in df_fe.columns if c not in original_cols]

print(f"Original features:  {len(original_cols)}")
print(f"After engineering:  {len(df_fe.columns)}")
print(f"New features added: {len(new_cols)}")
print()
print("New features:")
for col in new_cols:
    print(f"  ✓ {col}")


In [ ]:
# ── Feature Importance — Mutual Information ────────────────────────────────
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

target = 'isFraud'
feature_cols = get_feature_names(df_fe)
feature_cols = [c for c in feature_cols if c in df_fe.columns and df_fe[c].dtype in [np.float64, np.int64, np.uint8, bool, int, float]]

X_sample = df_fe[feature_cols].fillna(0)
y_sample = df_fe[target]

mi_scores = mutual_info_classif(X_sample, y_sample, random_state=42)
mi_df = pd.DataFrame({'Feature': feature_cols, 'MI_Score': mi_scores})
mi_df = mi_df.sort_values('MI_Score', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E15759' if s > mi_df['MI_Score'].quantile(0.75) else '#4E79A7' for s in mi_df['MI_Score']]
bars = ax.barh(mi_df['Feature'], mi_df['MI_Score'], color=colors, edgecolor='white', linewidth=0.3)
ax.set_xlabel('Mutual Information Score', color='white')
ax.set_title('Top 20 Features by Mutual Information with Fraud Label', color='white', fontsize=13, pad=12)
ax.grid(True, alpha=0.2, axis='x')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../visuals/03_feature_importance_mi.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


In [ ]:
# ── Feature Correlation Heatmap ───────────────────────────────────────────
# Select key numerical features
key_feats = [
    'amount', 'log_amount', 'balance_diff_orig', 'balance_diff_dest',
    'orig_balance_discrepancy', 'dest_balance_discrepancy',
    'amount_to_orig_balance_ratio', 'zero_balance_after_orig',
    'account_drained', 'is_night', 'is_high_risk_type', 'isFraud'
]
key_feats = [c for c in key_feats if c in df_fe.columns]

corr = df_fe[key_feats].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            linecolor='#0f1117', ax=ax, cbar_kws={'shrink': 0.7},
            annot_kws={'size': 8, 'color': 'white'})

ax.set_title('Feature Correlation Matrix', color='white', fontsize=14, pad=15)
ax.tick_params(labelcolor='white', labelsize=9)
plt.tight_layout()
plt.savefig('../visuals/03_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


In [ ]:
# ── New Feature Validation: Account Drained ───────────────────────────────
if 'account_drained' in df_fe.columns:
    drain_fraud_rate = df_fe.groupby('account_drained')['isFraud'].mean() * 100
    print("Fraud Rate by 'Account Drained' Feature:")
    print(f"  Account NOT drained → Fraud rate: {drain_fraud_rate.get(0, 0):.3f}%")
    print(f"  Account WAS drained → Fraud rate: {drain_fraud_rate.get(1, 0):.2f}%")
    lift = drain_fraud_rate.get(1, 0) / drain_fraud_rate.get(0, 1)
    print(f"  Lift: {lift:.1f}x more likely to be fraud when account is drained ✓")


## ✅ Feature Engineering Summary

| Feature Group | Features Created | Why It Matters |
|---|---|---|
| Balance discrepancy | `orig_balance_discrepancy`, `dest_balance_discrepancy` | Mismatches reveal hidden fund movements |
| Balance ratios | `amount_to_orig_balance_ratio` | Large fraction of balance = suspicious |
| Zero-balance flags | `zero_balance_after_orig`, `account_drained` | Account draining is a fraud red flag |
| Temporal | `hour_of_day`, `is_night`, `is_weekend` | Fraud peaks in off-hours |
| Type encoding | `is_high_risk_type`, one-hot types | Only TRANSFER/CASH-OUT have fraud |
| Amount features | `log_amount`, `is_round_amount`, `is_large_transaction` | Round numbers and large amounts signal fraud |

➡️ **Next:** Notebook 4 — Preprocessing & SMOTE
